📦 Bloque 1 — Importación de Librerías y Preparación del Entorno

Este bloque carga todas las librerías necesarias para trabajar con:
- Limpieza de texto y normalización (re, unicodedata)
- Manejo de rutas (Path)
- Fechas (datetime)
- Tipos y combinaciones (typing, itertools)
- Tratamiento de advertencias (warnings)
- Procesamiento de datos (pandas)
- Conexión a SQL Server (sqlalchemy)

Objetivo: Dejar preparado el entorno para las etapas de lectura y carga de datos.

In [ ]:
import pandas as pd
import re
import unicodedata
import itertools
import numpy as np
from datetime import datetime
from sqlalchemy import text
from typing import Tuple, Optional, Dict, List
import sys
import importlib
import json
import os
from os import path
from pathlib import Path

# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks/banigualdad/
# queremos:  .../traspaso-innominados/functions
#            .../traspaso-innominados/config
PROJECT_ROOT = Path(os.environ["PROJECT_ROOT"]).resolve()
FUNCTIONS_DIR = PROJECT_ROOT / "functions"
CONFIG_DIR = PROJECT_ROOT / "config"


print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())
print("CONFIG_DIR:",    CONFIG_DIR,    "exists:", CONFIG_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
module_name = "functions"
try:
    if module_name in sys.modules:
        fun = importlib.reload(sys.modules[module_name])
    else:
        fun = importlib.import_module(module_name)
    print("Módulo importado desde:", getattr(fun, "__file__", "<sin __file__>"))
except Exception as e:
    raise ImportError(
        f"No se pudo importar el módulo '{module_name}'.\n"
        f"Revisa que exista en: {FUNCTIONS_DIR}\n"
        f"Detalle: {e}"
    )

# --- Cargar config JSON ---
config_file = path.join(CONFIG_DIR, "banigualdad", "config_banigualdad.json")
try:
    with open(config_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("Config cargada desde:", config_file)
    print("Claves disponibles:", list(data.keys()))
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró el archivo de configuración:\n{config_file}\n"
        f"¿Existe CONFIG_DIR?: {CONFIG_DIR.exists()}"
    )
except json.JSONDecodeError as e:
    raise ValueError(f"Error: JSON inválido en {config_file}: {e}")
except Exception as e:
    raise RuntimeError(f"Error inesperado leyendo {config_file}: {e}")


🔌 Bloque 2 — Configuración de Conexión SQL Server

Define la información necesaria para conectarse a la base de datos:
- Servidor, base de datos, esquema y tabla.
- Construcción del connection_string con autenticación integrada.
- Creación del engine para ejecutar consultas y cargar datos.

Incluye mejoras de rendimiento y estabilidad como:
- fast_executemany=True
- pool_pre_ping=True

Objetivo: Establecer una conexión estable y optimizada con SQL Server.

In [ ]:
# --- Parámetros de conexión ---
server   = data["server_config"]["server"]
database = data["server_config"]["database"]
schema   = data["server_config"]["schema"]
table    = data["tablas_remotas"]["tabla_principal"]

driver = "ODBC Driver 17 for SQL Server"

# ODBC connection string (pyodbc) → para query_to_df / df_to_db / exec_sql
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# SQLAlchemy engine
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise"
)


📁 Bloque 3 — Configuración de Carpeta y Parámetros de Lectura CSV

Define la ruta donde se buscarán los archivos y los parámetros base que controlan:
- Extensiones válidas (.csv)
- Separadores posibles (;, ,, tab)
- Codificaciones típicas (utf-8-sig, latin1, etc.)
- Formato de fechas (día primero)
- Manejo de valores NA

Objetivo: Establecer reglas flexibles que permitan leer correctamente CSV con diferentes formatos.

In [ ]:
RUTA_ARCHIVOS = (PROJECT_ROOT / "data" / "banigualdad").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUTA_ARCHIVOS:", RUTA_ARCHIVOS, "exists:", RUTA_ARCHIVOS.exists())

EXTS = (".csv",)  # Solo CSV

DEFAULT_SEPARATORS = [";", ",", "\t"]
DEFAULT_ENCODINGS  = ["utf-8-sig", "utf-8", "latin1", "cp1252"]
DAYFIRST_DATES     = True
KEEP_DEFAULT_NA    = False


🧽 Bloque 4 — Normalización y Limpieza de Datos

Incluye funciones encargadas de estandarizar columnas y contenidos:
- _strip_accents: Elimina acentos.
- normalize_name: Convierte nombres a formato uniforme (snake_case).
- normalize_columns: Aplica normalización a todas las columnas.
- strip_strings: Limpia espacios y comillas en textos.
- coerce_dates: Convierte columnas a tipo fecha.
- coerce_numeric: Convierte montos en formato chileno a números reales.

Objetivo: Garantizar que los datos estén limpios, consistentes y listos para análisis o carga.

In [4]:
def _strip_accents(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s)) if not unicodedata.combining(c))


def normalize_name(s: str) -> str:
    s = _strip_accents(s).lower().strip()
    s = s.replace("°", "")
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^\w]", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [normalize_name(c) for c in df.columns]
    return df


def _norm(s):
    return " ".join(str(s).split()).lower()


def strip_strings(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip()
            # opcional: elimina comillas que a veces quedan en CSV
            df[c] = df[c].str.strip('"')
    return df


def coerce_dates(df: pd.DataFrame, cols: List[str], dayfirst: bool = True) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce", dayfirst=dayfirst)
    return df


def coerce_numeric(df: pd.DataFrame, cols: List[str], thousands: Optional[str] = ".", decimal: Optional[str] = ",") -> pd.DataFrame:
    """
    Convierte columnas a numérico respetando formatos con miles y decimales locales.
    Ej: "1.234.567,89" -> 1234567.89
    """
    df = df.copy()
    for c in cols:
        if c in df.columns:
            serie = df[c].astype(str)
            if thousands:
                serie = serie.str.replace(thousands, "", regex=False)
            if decimal and decimal != ".":
                serie = serie.str.replace(decimal, ".", regex=False)
            df[c] = pd.to_numeric(serie, errors="coerce")
    return df


def _extraer_fecha_archivo(nombre_archivo: str) -> datetime:
    """
    Busca un bloque de 8 dígitos (YYYYMMDD) en el nombre del archivo
    y lo convierte a datetime.
    Ej: 'remesa_20251104.csv' -> datetime(2025, 11, 4)
    """
    base = Path(nombre_archivo).name  # por si viene con ruta completa
    m = re.search(r"(\d{8})", base)
    if not m:
        raise ValueError(
            f"No se encontró una fecha YYYYMMDD en el nombre de archivo: {nombre_archivo}"
        )
    return datetime.strptime(m.group(1), "%Y%m%d")


def _yyyymm_mes_anterior(fecha: datetime) -> str:
    """
    A partir de una fecha, devuelve el YYYYMM del mes anterior.
    Maneja bien el cambio de año (enero -> diciembre del año anterior).
    """
    anio = fecha.year
    mes = fecha.month

    if mes == 1:
        mes = 12
        anio -= 1
    else:
        mes -= 1

    return f"{anio}{mes:02d}"


def _construir_nombre_archivo_remesa(nombre_archivo_fisico: str) -> str:
    """
    A partir del nombre real del archivo (ej: remesa_20251104.csv),
    construye el valor para la columna:
    remesa_YYYYMMAnterior.csv (ej: remesa_202510.csv)
    """
    fecha = _extraer_fecha_archivo(nombre_archivo_fisico)
    yyyymm_prev = _yyyymm_mes_anterior(fecha)
    return f"remesa_{yyyymm_prev}.csv"


def asegurar_columna_nombre_archivo_df_inicial(
    df: pd.DataFrame,
    archivo: Path,
    col_normalizada: str = "nombre_archivo",
) -> pd.DataFrame:
    """
    - Si la columna 'nombre_archivo' ya existe y tiene valores no vacíos, NO se toca.
    - Si no existe o está vacía, se crea usando el nombre del archivo físico:
      remesa_YYYYMMAnterior.csv (ej: remesa_202510.csv)
    """
    df = df.copy()

    if col_normalizada in df.columns:
        serie = df[col_normalizada].astype("string").str.strip()
        vals = [v for v in serie.dropna().unique() if str(v).strip() != ""]
        if vals:
            print("ℹ️ Se detectó columna 'nombre_archivo' con valores. Se respeta lo que viene en el archivo.")
            return df

    valor = _construir_nombre_archivo_remesa(archivo.name)
    df[col_normalizada] = valor
    print(f"🆕 Columna 'nombre_archivo' creada a partir del nombre físico: {valor}")
    return df




📥 Bloque 5 — Lector Robusto de CSV (read_csv_robusto)

Función avanzada que permite leer archivos CSV aunque tengan variaciones en su formato.

Incluye:
- Prueba automática de múltiples codificaciones y separadores.
- Lectura completa como texto para evitar pérdida de información.
- Normalización de columnas.
- Limpieza de textos.
- Conversión automática de:
    - Fechas
    - Numéricos
    - Tipos según pistas (dtype_hints)
- Eliminación de filas completamente vacías.

Objetivo: Leer correctamente el archivo sin importar cómo venga formateado.

In [5]:
def read_csv_robusto(
    archivo: Path,
    encodings=DEFAULT_ENCODINGS,
    seps=DEFAULT_SEPARATORS,
    keep_default_na: bool = KEEP_DEFAULT_NA,
    dtype_hints: Dict[str, str] = None,
    date_cols: List[str] = None,
    numeric_cols: List[str] = None,
    dayfirst: bool = DAYFIRST_DATES,
) -> pd.DataFrame:
    """
    Lee CSV probando encodings y separadores.
    - Carga inicialmente dtype=str para no perder información.
    - Luego normaliza columnas, trimea strings y castea por pistas (hints).
    """
    dtype_hints = dtype_hints or {}
    date_cols = date_cols or []
    numeric_cols = numeric_cols or []

    last_err = None
    for enc, sep in itertools.product(encodings, seps):
        try:
            df = pd.read_csv(
                archivo,
                sep=sep,
                encoding=enc,
                dtype=str,
                keep_default_na=keep_default_na,
                engine="python",  # mejor tolerancia con CSV "difíciles"
            )
            # Si leyó vacío y no era esperado, intentamos siguiente combinación
            if df.shape[1] <= 1 and sep != ",":
                # quizá el separador no era el correcto; seguimos probando
                continue

            print(f"✅ CSV leído con encoding='{enc}' y separador='{sep}'. Shape: {df.shape}")
            # Limpiezas y casteos
            df = normalize_columns(df)
            df = strip_strings(df)

            # Aplica hints de tipo
            # 1) fechas
            if date_cols:
                df = coerce_dates(df, date_cols, dayfirst=dayfirst)

            # 2) numéricos
            if numeric_cols:
                df = coerce_numeric(df, numeric_cols, thousands=".", decimal=",")

            # 3) hints dtype explícitos (string, float64, int64, etc.)
            for col, hint in (dtype_hints or {}).items():
                if col in df.columns and hint not in ("date",):
                    try:
                        if hint.lower() in ("string", "str", "object"):
                            df[col] = df[col].astype("string")
                        elif hint.lower() in ("float", "float64", "double"):
                            df[col] = pd.to_numeric(df[col], errors="coerce")
                        elif hint.lower() in ("int", "int64"):
                            # cuidado con NaN → usar Int64 (nullable)
                            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
                        else:
                            # fallback: intentar astype directo
                            df[col] = df[col].astype(hint)
                    except Exception as _e:
                        warnings.warn(f"No se pudo castear '{col}' a '{hint}': {_e}")

            df.dropna(how="all", inplace=True)

            return df

        except Exception as e:
            last_err = e
            continue

    raise SystemExit(f"❌ No se pudo leer el CSV con las combinaciones dadas. Último error: {last_err}")

📄 Bloque 6 — Selección del Archivo Más Reciente y Lectura

Acciones principales:
- Verifica que la carpeta exista.
- Obtiene todos los CSV válidos.
- Ordena por fecha de modificación.
- Selecciona el archivo más reciente.
- Llama al lector robusto para obtener un DataFrame limpio.
- Muestra información del archivo y del DataFrame.

Objetivo: Automatizar la elección del archivo correcto y obtener los datos preprocesados.

In [6]:
assert RUTA_ARCHIVOS.is_dir(), f"No existe carpeta: {RUTA_ARCHIVOS}"

archivos = [p for p in RUTA_ARCHIVOS.iterdir() if p.is_file() and p.suffix.lower() in EXTS]
archivos.sort(key=lambda p: p.stat().st_mtime, reverse=True)
assert archivos, "No se encontraron archivos CSV válidos."

archivo = archivos[0]
print(f"📄 Archivo más reciente: {archivo.name}  |  {datetime.fromtimestamp(archivo.stat().st_mtime):%Y-%m-%d %H:%M:%S}")

df = read_csv_robusto(
    archivo=archivo,
)

df = asegurar_columna_nombre_archivo_df_inicial(df, archivo)

print(f"✅ DataFrame final: {df.shape[0]} filas x {df.shape[1]} columnas")
print("🔎 Columnas:", ", ".join(df.columns[:25]) + (" ..." if df.shape[1] > 25 else ""))


📄 Archivo más reciente: remesa_20251203 (2).csv  |  2026-01-12 15:47:10
✅ CSV leído con encoding='utf-8-sig' y separador=';'. Shape: (45660, 6)
🆕 Columna 'nombre_archivo' creada a partir del nombre físico: remesa_202511.csv
✅ DataFrame final: 45660 filas x 7 columnas
🔎 Columnas: numero_propuesta, fecha_de_pago, rut_afiliado, nombre_afiliado, n_cuota, valor_cuota_bruto_pesos, nombre_archivo


In [7]:
df.head()

,numero_propuesta,fecha_de_pago,rut_afiliado,nombre_afiliado,n_cuota,valor_cuota_bruto_pesos,nombre_archivo
0,1972255,20251126,15176707-9,Katherinne Alejandra Cerda Escobar,12,759,remesa_202511.csv
1,1972249,20251126,15553087-1,Mario Andres Mac Donald Mac Donald,12,759,remesa_202511.csv
2,1978525,20251126,16382512-0,Silvana Maritza Fuentes Leiva,12,759,remesa_202511.csv
3,1972245,20251126,18368253-9,Francisco Javier Pinilla Salamanca,12,759,remesa_202511.csv
4,1972252,20251126,9993215-5,Maria Victoria Del Campo Gonzalez,12,759,remesa_202511.csv


🧩 Bloque 7 — Mapeo y Construcción del DataFrame SQL

Función pick que selecciona la primera columna disponible entre varios nombres posibles.

Permite tolerar cambios en los nombres de columnas fuente.

Extrae columnas necesarias como:
- numero_propuesta
- fecha_de_pago
- rut_afiliado
- valor_cuota_bruto_pesos

Construye df_sql con los nombres finales requeridos por SQL Server.

Objetivo: Traducir el archivo fuente a la estructura estandarizada utilizada en la tabla SQL.

In [8]:
def pick(df, *names):
    for n in names:
        if n in df.columns:
            return df[n]
    return pd.Series([None]*len(df), index=df.index)


# Origen → Destino
numero_propuesta            = pick(df, "numero_propuesta")
fecha_de_pago               = pick(df, "fecha_de_pago")
rut_afiliado                = pick(df, "rut_afiliado")
nombre_afiliado             = pick(df, "nombre_afiliado")
n_cuota                     = pick(df, "n_cuota")
valor_cuota_bruto_pesos     = pick(df, "valor_cuota_bruto_pesos")
nombre_archivo              = pick(df, "nombre_archivo")



df_sql = pd.DataFrame({
    "PROPUESTA": numero_propuesta,
    "FECHA_PAGO": fecha_de_pago,
    "RUTAFILIADO": rut_afiliado,
    "NOMBREAFILIADO": nombre_afiliado,
    "CUOTA": n_cuota,
    "VALOR_CUOTA_BRUTO_CLP": valor_cuota_bruto_pesos,
    "NOMBRE_ARCHIVO": nombre_archivo,
})

df_sql.head()

,PROPUESTA,FECHA_PAGO,RUTAFILIADO,NOMBREAFILIADO,CUOTA,VALOR_CUOTA_BRUTO_CLP,NOMBRE_ARCHIVO
0,1972255,20251126,15176707-9,Katherinne Alejandra Cerda Escobar,12,759,remesa_202511.csv
1,1972249,20251126,15553087-1,Mario Andres Mac Donald Mac Donald,12,759,remesa_202511.csv
2,1978525,20251126,16382512-0,Silvana Maritza Fuentes Leiva,12,759,remesa_202511.csv
3,1972245,20251126,18368253-9,Francisco Javier Pinilla Salamanca,12,759,remesa_202511.csv
4,1972252,20251126,9993215-5,Maria Victoria Del Campo Gonzalez,12,759,remesa_202511.csv


🛡️ Bloque 8 — Verificación de NOMBRE_ARCHIVO y Control de Cargas Previas

Este bloque garantiza que solo se carguen archivos válidos y que la carga sea segura:

- Verifica que la columna NOMBRE_ARCHIVO exista en el DataFrame.
- Normaliza sus valores (quita espacios, controla vacíos).
- Identifica todos los valores únicos (útil para consolidados con varios meses).
- Muestra un conteo por cada archivo detectado.

Prepara la variable vals, que será usada para decidir:

- Si cada archivo será insertado, o si será reemplazado en SQL Server.

In [9]:
assert "NOMBRE_ARCHIVO" in df_sql.columns, "Falta la columna 'NOMBRE_ARCHIVO' en df_sql."

df_sql["NOMBRE_ARCHIVO"] = (
    df_sql["NOMBRE_ARCHIVO"]
    .astype("string")
    .str.strip()
)


vals = [
    v for v in df_sql["NOMBRE_ARCHIVO"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚨 No se encontraron valores válidos en 'NOMBRE_ARCHIVO'.")

counts = (
    df_sql["NOMBRE_ARCHIVO"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"📄 Se detectaron {len(vals)} NOMBRE_ARCHIVO distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

📄 Se detectaron 1 NOMBRE_ARCHIVO distintos en el df_sql:
   - remesa_202511.csv: 45660 filas


🚀 Bloque 9 — Carga Dinámica a SQL Server + Reemplazo Inteligente

Este bloque realiza la carga final al Data Warehouse de forma controlada y segura.

Para cada NOMBRE_ARCHIVO detectado:

- Verifica si ya existe en SQL Server.    
- Si existe → elimina esas filas.   
- Inserta los datos nuevos del archivo (o consolidado).   
- Todo dentro de una sola transacción, evitando inconsistencias.    
- Genera un resumen detallado por archivo: reemplazo o inserción nueva.

In [10]:
resumen = []

with engine.begin() as conn:
    qry_count = text(f"""
        SELECT COUNT(*) AS cnt
        FROM {schema}.{table}
        WHERE NOMBRE_ARCHIVO = :nombre
    """)

    qry_del = text(f"""
        DELETE FROM {schema}.{table}
        WHERE NOMBRE_ARCHIVO = :nombre
    """)

    for nombre_archivo in vals:
        df_sub = df_sql[df_sql["NOMBRE_ARCHIVO"] == nombre_archivo]

        if df_sub.empty:
            print(f"⚠️ No se encontraron filas en df_sql para NOMBRE_ARCHIVO = '{nombre_archivo}'. Se omite.")
            continue

        existing_count = conn.execute(qry_count, {"nombre": nombre_archivo}).scalar() or 0

        if existing_count > 0:
            print(f"♻️ Se encontró data previa para NOMBRE_ARCHIVO = '{nombre_archivo}' "
                  f"({existing_count} filas en {schema}.{table}).")
            print("🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...")

            deleted = conn.execute(qry_del, {"nombre": nombre_archivo}).rowcount
            print(f"✅ Filas eliminadas en destino para '{nombre_archivo}': {deleted}")
            accion = "reemplazo"
        else:
            print(f"🆕 No hay data previa para NOMBRE_ARCHIVO = '{nombre_archivo}'. "
                  "Se insertará como archivo nuevo.")
            accion = "inserción nueva"

        df_sub.to_sql(
            name=table,
            con=conn,
            schema=schema,
            if_exists='append',
            index=False
        )

        filas_sub = len(df_sub)
        print(f"📥 Insertadas {filas_sub} filas para NOMBRE_ARCHIVO = '{nombre_archivo}'.")
        resumen.append((nombre_archivo, filas_sub, existing_count, accion))


print("\n📊 Resumen de carga por NOMBRE_ARCHIVO:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (reemplazando {existing_count} previas).")
    else:
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (archivo nuevo).")

♻️ Se encontró data previa para NOMBRE_ARCHIVO = 'remesa_202511.csv' (45660 filas en dbo.BANIGUALDAD_HISTORICO_R).
🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...
✅ Filas eliminadas en destino para 'remesa_202511.csv': 45660
📥 Insertadas 45660 filas para NOMBRE_ARCHIVO = 'remesa_202511.csv'.

📊 Resumen de carga por NOMBRE_ARCHIVO:
   - remesa_202511.csv: 45660 filas cargadas (reemplazando 45660 previas).


🗑️ Bloque 10 – Eliminación del archivo procesado

Finalmente:
- Una vez cargado en SQL, el archivo fuente se elimina automáticamente.
- Maneja errores comunes:
    - Archivo en uso por Excel/OneDrive
    - Permisos insuficientes
    - Casos en que el archivo no existe
    
Cierra el ciclo ETL manteniendo la carpeta siempre limpia.

In [ ]:
try:
    if archivo.exists() and archivo.is_file():
        archivo.unlink()
        print(f"\n🗑️ Archivo eliminado correctamente: {archivo}")
    else:
        print(f"\n⚠️ No se pudo borrar el archivo porque no es un archivo válido: {archivo}")
except PermissionError:
    print(f"\n⚠️ No se pudo borrar '{archivo}': está en uso por OneDrive o Excel.")
except Exception as e:
    print(f"\n⚠️ Error inesperado al borrar archivo '{archivo}': {e}")